# 08 — Reactive Basics

`data_setter`, `data_formula`, `subscribe()` — the reactive engine.

**What you learn:**
- `data_setter`: declare a static value in the reactive store
- `data_formula`: declare a computed value that re-executes when deps change
- `ReactiveManager` with `subscribe=True` activates reactive bindings
- Formula dependency chains are resolved via topological sort
- Changing data after subscribe automatically updates output

**Prerequisites:** 04_pointers

In [ ]:
from IPython.display import HTML
from genro_builders.contrib.html import HtmlBuilder
from genro_builders.manager import ReactiveManager


class PriceCalculator(ReactiveManager):
    def __init__(self):
        self.page = self.set_builder('page', HtmlBuilder)
        self.run(subscribe=True)

    def store(self, data):
        data['base_price'] = 100
        data['discount'] = 0.1
        data['tax_rate'] = 0.22

    def main(self, source):
        source.data_setter('base_price', value='^base_price')
        source.data_setter('discount', value='^discount')
        source.data_setter('tax_rate', value='^tax_rate')

        # Computed: net_price depends on base_price and discount
        source.data_formula(
            'net_price',
            func=lambda base_price, discount: base_price * (1 - discount),
            base_price='^base_price', discount='^discount',
        )
        # Computed: total depends on net_price and tax_rate
        source.data_formula(
            'total',
            func=lambda net_price, tax_rate: round(net_price * (1 + tax_rate), 2),
            net_price='^net_price', tax_rate='^tax_rate',
        )

        source.head().style('body { font-family: sans-serif; max-width: 400px; margin: 2em auto; } .result { font-size: 1.3em; color: #16a34a; font-weight: bold; }')
        body = source.body()
        body.h1('Price Calculator')
        body.p('^base_price')
        body.p('^net_price', _class='result')
        body.p('^total', _class='result')

In [ ]:
app = PriceCalculator()
store = app.reactive_store
print(f'Initial: base={store["base_price"]}, net={store["net_price"]}, total={store["total"]}')
HTML(app.page.render())

## Change data — formulas re-execute automatically

In [ ]:
store['base_price'] = 200
print(f'After change: base={store["base_price"]}, net={store["net_price"]}, total={store["total"]}')
HTML(app.page.render())

In [ ]:
store['discount'] = 0.25
print(f'After discount: base={store["base_price"]}, net={store["net_price"]}, total={store["total"]}')
HTML(app.page.render())